# Project Setup

In this notebook, we'll import our datasets, wrangle and clean them, and then store them in a parquet file that we can use in the next stage of the project.

## Library Import

In [44]:
# Common DataFrame Tools
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Python Libraries
from datetime import datetime
import json

## Dataset Import

In [5]:
df_a = pd.read_json('../../raw-data/Kickstarter_2026-07-24T08_26_14_228Z.json', lines=True)

In [6]:
df_a.head(2)

,table_id,robot_id,run_id,data
0,Kickstarter,Kickstarter,Kickstarter_2026-07-24T08_26_14_228Z,"{'id': 1703332404, 'photo': {'key': 'assets/04..."
1,Kickstarter,Kickstarter,Kickstarter_2026-07-24T08_26_14_228Z,"{'id': 1704943227, 'photo': {'key': 'assets/04..."


In [7]:
df_a.shape

(112499, 4)

In [8]:
df_b = pd.read_csv('../../raw-data/kickstarter_data_full.csv')

/tmp/ipykernel_2251121/3050664646.py:1: DtypeWarning: Columns (29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df_b = pd.read_csv('../../raw-data/kickstarter_data_full.csv')


We see mixed types on the indicated rows. Let's see which those are.

In [10]:
column_list = list(enumerate(df_b.columns))

In [11]:
column_list

[(0, 'Unnamed: 0'),
 (1, 'id'),
 (2, 'photo'),
 (3, 'name'),
 (4, 'blurb'),
 (5, 'goal'),
 (6, 'pledged'),
 (7, 'state'),
 (8, 'slug'),
 (9, 'disable_communication'),
 (10, 'country'),
 (11, 'currency'),
 (12, 'currency_symbol'),
 (13, 'currency_trailing_code'),
 (14, 'deadline'),
 (15, 'state_changed_at'),
 (16, 'created_at'),
 (17, 'launched_at'),
 (18, 'staff_pick'),
 (19, 'backers_count'),
 (20, 'static_usd_rate'),
 (21, 'usd_pledged'),
 (22, 'creator'),
 (23, 'location'),
 (24, 'category'),
 (25, 'profile'),
 (26, 'spotlight'),
 (27, 'urls'),
 (28, 'source_url'),
 (29, 'friends'),
 (30, 'is_starred'),
 (31, 'is_backing'),
 (32, 'permissions'),
 (33, 'name_len'),
 (34, 'name_len_clean'),
 (35, 'blurb_len'),
 (36, 'blurb_len_clean'),
 (37, 'deadline_weekday'),
 (38, 'state_changed_at_weekday'),
 (39, 'created_at_weekday'),
 (40, 'launched_at_weekday'),
 (41, 'deadline_month'),
 (42, 'deadline_day'),
 (43, 'deadline_yr'),
 (44, 'deadline_hr'),
 (45, 'state_changed_at_month'),
 (46, '

In [14]:
column_list[29:33]

[(29, 'friends'), (30, 'is_starred'), (31, 'is_backing'), (32, 'permissions')]

In [16]:
temp_df = df_b[['friends', 'is_starred', 'is_backing', 'permissions']]

In [17]:
temp_df.head(2)

,friends,is_starred,is_backing,permissions
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN


In [18]:
temp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20632 entries, 0 to 20631
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   friends      60 non-null     object
 1   is_starred   60 non-null     object
 2   is_backing   60 non-null     object
 3   permissions  60 non-null     object
dtypes: object(4)
memory usage: 644.9+ KB


In [19]:
temp_df.describe()

,friends,is_starred,is_backing,permissions
count,60,60,60,60
unique,1,1,1,1
top,[],False,False,[]
freq,60,60,60,60


In these columns, there are only `60` values. Are they worth maintaining in our overall dataset? Or should we simply delete them?

In [20]:
df_b.shape

(20632, 68)

Let's try avoiding these columns on import and see if we still get the same warning.

In [21]:
df_b = pd.read_csv('../../raw-data/kickstarter_data_full.csv', usecols=lambda x: x not in temp_df.columns)

Great, we don't see an error now.

In [22]:
df_b.head(2)

,Unnamed: 0,id,photo,name,blurb,goal,pledged,state,slug,disable_communication,...,launch_to_deadline,launch_to_state_change,create_to_launch_days,launch_to_deadline_days,launch_to_state_change_days,SuccessfulBool,USorGB,TOPCOUNTRY,LaunchedTuesday,DeadlineWeekend
0,0,1454391034,"{""small"":""https://ksr-ugc.imgix.net/assets/011...",Auntie Di's Music Time Sign ASL for Hearing an...,MTS ASL Curriculum Workbook is a reproducible ...,1500.0,0.0,failed,auntie-dis-music-time-sign-asl-for-hearing-and...,False,...,36 days 20:47:24.000000000,36 days 20:47:24.000000000,17,36,36,0,1,1,0,0
1,1,1655206086,"{""small"":""https://ksr-ugc.imgix.net/assets/012...",Jump Start Kindergarten Toolkit,"This kit teaches how to print, correct an ugly...",500.0,0.0,failed,jump-start-kindergarten-toolkit,False,...,60 days 00:00:00.000000000,60 days 00:00:02.000000000,10,60,60,0,1,1,0,0


### A Note on Filtering

This note is at the bottom of the Web Robots page:

> Note: from April 2015 we noticed that Kickstarter started limiting how many projects user can view in a single category. This limits the amount of historic projects we can get in a single scrape run. But recent and active projects are always included.

> Note: from December 2015 we modified the collection approach to go through all sub-categories instead of only top level categories. This yields more results in the datasets, but possible duplication where projects are listed in multiple categories. Also from December 2015 JSON file is in JSON streaming format. Read more about it here: https://en.wikipedia.org/wiki/JSON_Streaming

> We receive many question about timestamp format used in this dataset. It is unix time. Google has a lot of information about it.

> Warning: files are compressed, size in area of 100mb. Uncompressed size around 600mb.

With this warning, we can expect that the two datasets may not entirely overlap, since they both would have been scraped on different dates.

## Initial Inspection

Can the datasets be joined? Do they have unique data, or does `df_a` have all of the data that `df_b` contains?

In [23]:
df_a.columns

Index(['table_id', 'robot_id', 'run_id', 'data'], dtype='object')

In [27]:
df_a.iloc[0]

table_id                                          Kickstarter
robot_id                                          Kickstarter
run_id                   Kickstarter_2026-07-24T08_26_14_228Z
data        {'id': 1703332404, 'photo': {'key': 'assets/04...
Name: 0, dtype: object

In [28]:
df_a.iloc[1]

table_id                                          Kickstarter
robot_id                                          Kickstarter
run_id                   Kickstarter_2026-07-24T08_26_14_228Z
data        {'id': 1704943227, 'photo': {'key': 'assets/04...
Name: 1, dtype: object

In [31]:
df_a.iloc[:,0:3]

,table_id,robot_id,run_id
0,Kickstarter,Kickstarter,Kickstarter_2026-07-24T08_26_14_228Z
1,Kickstarter,Kickstarter,Kickstarter_2026-07-24T08_26_14_228Z
2,Kickstarter,Kickstarter,Kickstarter_2026-07-24T08_26_14_228Z
3,Kickstarter,Kickstarter,Kickstarter_2026-07-24T08_26_14_228Z
4,Kickstarter,Kickstarter,Kickstarter_2026-07-24T08_26_14_228Z
...,...,...,...
112494,Kickstarter,Kickstarter,Kickstarter_2026-07-24T08_26_14_228Z
112495,Kickstarter,Kickstarter,Kickstarter_2026-07-24T08_26_14_228Z
112496,Kickstarter,Kickstarter,Kickstarter_2026-07-24T08_26_14_228Z
112497,Kickstarter,Kickstarter,Kickstarter_2026-07-24T08_26_14_228Z


These first three columns in `df_a` appear to be identical. This is likely a holdover from the `Web Robots` archive service from which we pulled this data.

Let's go ahead and drop it.

In [32]:
df_a = df_a.drop(columns=['table_id', 'robot_id', 'run_id'])

In [35]:
df_a.head(2)

,data
0,"{'id': 1703332404, 'photo': {'key': 'assets/04..."
1,"{'id': 1704943227, 'photo': {'key': 'assets/04..."


## Parsing the Json Objects

Let's take a look at a json object.

In [34]:
df_a.iloc[0,0]

{'id': 1703332404,
 'photo': {'key': 'assets/047/794/907/72d3766ab073b662350a131c856f3754_original.jpg',
  'full': 'https://i.kickstarter.com/assets/047/794/907/72d3766ab073b662350a131c856f3754_original.jpg?anim=false&fit=cover&gravity=auto&height=315&origin=ugc&q=92&v=1736543445&width=560&sig=FOKcMyVwa6p%2FmRcvu2vSc7y9RvC6EKAVeg82naZokEA%3D',
  'ed': 'https://i.kickstarter.com/assets/047/794/907/72d3766ab073b662350a131c856f3754_original.jpg?anim=false&fit=cover&gravity=auto&height=198&origin=ugc&q=92&v=1736543445&width=352&sig=mY5wv1R5hgBTCm2bQfX4Qt%2BCmHZv212%2BfR99NlKgqss%3D',
  'med': 'https://i.kickstarter.com/assets/047/794/907/72d3766ab073b662350a131c856f3754_original.jpg?anim=false&fit=cover&gravity=auto&height=153&origin=ugc&q=92&v=1736543445&width=272&sig=gqkYSwkzFgpCdjXWaIYn1aGNga0bv7qIpAWU4R4eu4s%3D',
  'little': 'https://i.kickstarter.com/assets/047/794/907/72d3766ab073b662350a131c856f3754_original.jpg?anim=false&fit=cover&gravity=auto&height=117&origin=ugc&q=92&v=17365434

Taking a look through here, we can see that this json data is a campaign for a Massage center based on donations. 

Here are the dates related to this event.

In [41]:
temp = []
# 'deadline': 
temp.append(datetime.fromtimestamp(1743442361))
# result time
temp.append(datetime.fromtimestamp(1743442365))
# Project created
temp.append(datetime.fromtimestamp(1736541915))
# Launched
temp.append(datetime.fromtimestamp(1738261961))

In [43]:
print(temp[0:4])

[datetime.datetime(2025, 3, 31, 9, 32, 41), datetime.datetime(2025, 3, 31, 9, 32, 45), datetime.datetime(2025, 1, 10, 11, 45, 15), datetime.datetime(2025, 1, 30, 9, 32, 41)]


This event was created in January of 2025, ran for approximately 60 days, and resulted in a failed campaign at the end of March of the same year.

A question I have after observing this is, are all of these json objects from the same timeframe? Or is our dataset all inclusive of the Kickstarter project library?

Our next natural step is to create a json parser that can filter this content into a new dataframe. We have a few items that are nested, so we will need to break those items into unique columns.

In [49]:
df_a_parse = pd.json_normalize(df_a['data'])

In [50]:
df_a_parse.head(2)

,id,name,blurb,goal,pledged,state,slug,country,country_displayable_name,currency,...,video.base_type,video.tracks,video.width,video.height,video.frame,profile.background_image_attributes.id,profile.background_image_attributes.image_urls.default,profile.background_image_attributes.image_urls.baseball_card,profile.feature_image_attributes.id,creator.is_ksr_admin
0,1703332404,Massage By Donation... Seriously!,Physical wellness is not a luxury; it is a nec...,53170.0,52.0,failed,massage-by-donation-seriously,US,the United States,USD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1704943227,eifel-abenteuer.de - Nachhaltige Outdoorangebo...,Aufbau eines erlebnispädagogischen Outdoorakti...,10000.0,0.0,failed,eifel-abenteuerde-nachhaltige-outdoorangebote-...,DE,Germany,EUR,...,"video/mp4; codecs=""avc1.42E01E, mp4a.40.2""",[],640.0,360.0,https://d15chbti7ht62o.cloudfront.net/projects...,NaN,NaN,NaN,NaN,NaN


In [60]:
temp_cols = pd.DataFrame(df_a_parse.columns)

In [63]:
with pd.option_context('display.max_rows', None):
    print(temp_cols)

                                                     0
0                                                   id
1                                                 name
2                                                blurb
3                                                 goal
4                                              pledged
5                                                state
6                                                 slug
7                                              country
8                             country_displayable_name
9                                             currency
10                                     currency_symbol
11                              currency_trailing_code
12                                            deadline
13                                    state_changed_at
14                                          created_at
15                                         launched_at
16                  is_in_post_campaign_pledging_phase
17        

### Perusing the Json Dataset

Lets see what the oldest project is in this dataset.

In [66]:
print(datetime.fromtimestamp(df_a_parse['created_at'].min()))

2009-04-22 19:06:59


Success. As we can see here, this project came from 2009, which either is the oldest project created, or is at the very least old enough for our purposes. Out of curiosity, let's take a look at the details from this project.

In [87]:
# Retrieve the minimum index
tempMin = df_a_parse['created_at'].idxmin(axis=0)

In [81]:
tempMin

110777

In [85]:
df_a_parse.iloc[tempMin]['creator.urls.web.user']

'https://www.kickstarter.com/profile/1782188740'

In [86]:
df_a_parse.iloc[tempMin]['urls.web.project']

'https://www.kickstarter.com/projects/1782188740/how-to-build-a-city-in-200-days-mahabalipuram-203'

After a little bit of hunting and tweaking, we find that the `urls.web.project` column contains the web location for the project.

This first recorded project was a city plan for India. The project was a success.

### Checking for Join Capabilities

Our next question is whether or not these datasets can be combined. Each as an `id` column. Do they match? 

As we noted earlier, the datasets may not match, because Web Robots is limited by Kickstarter's search methods.

In [91]:
temp_id = df_b.loc[0, 'id']
temp_name = df_b.loc[0, 'name']

In [93]:
df_a_parse[df_a_parse['id'] == temp_id]

,id,name,blurb,goal,pledged,state,slug,country,country_displayable_name,currency,...,video.base_type,video.tracks,video.width,video.height,video.frame,profile.background_image_attributes.id,profile.background_image_attributes.image_urls.default,profile.background_image_attributes.image_urls.baseball_card,profile.feature_image_attributes.id,creator.is_ksr_admin


No luck so far.

In [94]:
df_a_parse[df_a_parse['name'] == temp_name]

,id,name,blurb,goal,pledged,state,slug,country,country_displayable_name,currency,...,video.base_type,video.tracks,video.width,video.height,video.frame,profile.background_image_attributes.id,profile.background_image_attributes.image_urls.default,profile.background_image_attributes.image_urls.baseball_card,profile.feature_image_attributes.id,creator.is_ksr_admin


In [95]:
df_b.loc[0]

Unnamed: 0                                                         0
id                                                        1454391034
photo              {"small":"https://ksr-ugc.imgix.net/assets/011...
name               Auntie Di's Music Time Sign ASL for Hearing an...
blurb              MTS ASL Curriculum Workbook is a reproducible ...
                                         ...                        
SuccessfulBool                                                     0
USorGB                                                             1
TOPCOUNTRY                                                         1
LaunchedTuesday                                                    0
DeadlineWeekend                                                    0
Name: 0, Length: 64, dtype: object

In [96]:
df_a_parse[df_a_parse['name'].str.contains('Auntie Di')]

,id,name,blurb,goal,pledged,state,slug,country,country_displayable_name,currency,...,video.base_type,video.tracks,video.width,video.height,video.frame,profile.background_image_attributes.id,profile.background_image_attributes.image_urls.default,profile.background_image_attributes.image_urls.baseball_card,profile.feature_image_attributes.id,creator.is_ksr_admin


In [97]:
df_b.shape

(20632, 64)

In [98]:
temp_id = df_b.loc[20631, 'id']
temp_name = df_b.loc[20631, 'name']

In [99]:
df_a_parse[df_a_parse['id'] == temp_id]

,id,name,blurb,goal,pledged,state,slug,country,country_displayable_name,currency,...,video.base_type,video.tracks,video.width,video.height,video.frame,profile.background_image_attributes.id,profile.background_image_attributes.image_urls.default,profile.background_image_attributes.image_urls.baseball_card,profile.feature_image_attributes.id,creator.is_ksr_admin


In [100]:
temp_name

"Let's Trail app"

A search in the search bar on Kickstarter shows that this is a real project.

In [103]:
df_a_parse[df_a_parse['name'].str.contains('s Trail')]

,id,name,blurb,goal,pledged,state,slug,country,country_displayable_name,currency,...,video.base_type,video.tracks,video.width,video.height,video.frame,profile.background_image_attributes.id,profile.background_image_attributes.image_urls.default,profile.background_image_attributes.image_urls.baseball_card,profile.feature_image_attributes.id,creator.is_ksr_admin
2169,1368143895,Endless Trail Digital Miniatures,STL files for 3d printing 32mm scale tabletop ...,1200.0,1872.00,successful,endless-trail-digital-miniatures,US,the United States,USD,...,"video/mp4; codecs=""avc1.42E01E, mp4a.40.2""",[],1280.0,720.0,https://d15chbti7ht62o.cloudfront.net/projects...,NaN,NaN,NaN,NaN,NaN
10735,2057048678,The Sovereign of the Seas: The Four Keys Trailer,We need YOU to help make a movie trailer for t...,4500.0,4768.00,successful,the-sovereign-of-the-seas-the-four-keys-trailer,US,the United States,USD,...,"video/mp4; codecs=""avc1.42E01E, mp4a.40.2""",[],640.0,360.0,https://d15chbti7ht62o.cloudfront.net/projects...,NaN,NaN,NaN,NaN,NaN
18141,419253913,Forbes Trail Brewing,A microbrewery with a production and taproom s...,5000.0,15867.00,successful,forbes-trail-brewing,US,the United States,USD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95128,1301255152,Touched By An Angel- Series Trailer,Let's reboot this incredible TV Show!,3200.0,3241.00,successful,touched-by-angel-series-trailer,US,the United States,USD,...,"video/mp4; codecs=""avc1.42E01E, mp4a.40.2""",[],640.0,360.0,https://d15chbti7ht62o.cloudfront.net/projects...,NaN,NaN,NaN,NaN,NaN
100543,159241682,Chicago Shadows Trailer,"A trailer for ""Chicago Shadows,"" our planned c...",15000.0,16104.08,successful,chicago-shadows-trailer,US,the United States,USD,...,"video/mp4; codecs=""avc1.42E01E, mp4a.40.2""",[],640.0,360.0,https://d15chbti7ht62o.cloudfront.net/projects...,NaN,NaN,NaN,NaN,NaN


In [105]:
df_a_parse.shape

(112499, 120)

In [106]:
df_b.shape

(20632, 64)

Projects from both datasets are showing up on the website.

Let's see if any of the ids from the smaller set show up in the larger set.

In [107]:
temp_id_list = df_b['id'].to_numpy()

In [135]:
df_a_parse[df_a_parse['id'].isin(temp_id_list)].shape

(2952, 120)

We see `2952` matching ids between these two sets.

Let's see if these ids actually match, or whether something else is happening.

In [110]:
temp_df = df_a_parse[df_a_parse['id'].isin(temp_id_list)][['id', 'name']] 

In [112]:
temp_join_df = pd.merge(temp_df, df_b[['id', 'name']], on='id', how='left')

In [113]:
temp_join_df.head()

,id,name_x,name_y
0,2146072954,Live 4 The Rush: Palooza Pics,Live 4 The Rush: Palooza Pics
1,1761289973,Travels Off Track,Travels Off Track
2,159506186,Paintbox Studio,Paintbox Studio
3,1895204636,Real Estate 3D Photos & Historical Structure D...,Real Estate 3D Photos & Historical Structure D...
4,785046586,Ours,Ours


That answers one question. These sets have some overlap. 

Let's take a look at the list of columns in both sets to see if we even want to join them.

In [118]:
temp_a = pd.DataFrame(df_a_parse.columns)
temp_b = pd.DataFrame(df_b.columns)

In [126]:
temp_a.columns = ['col_list']
temp_b.columns = ['col_list']

In [130]:
temp_a[temp_a['col_list'].isin(temp_b['col_list'] == True)]

,col_list


In [131]:
temp_a['col_list'].isin(temp_b['col_list'])

0       True
1       True
2       True
3       True
4       True
       ...  
115    False
116    False
117    False
118    False
119    False
Name: col_list, Length: 120, dtype: bool

In [132]:
with pd.option_context('display.max_rows', None):
    print(pd.DataFrame(df_a_parse.columns))

                                                     0
0                                                   id
1                                                 name
2                                                blurb
3                                                 goal
4                                              pledged
5                                                state
6                                                 slug
7                                              country
8                             country_displayable_name
9                                             currency
10                                     currency_symbol
11                              currency_trailing_code
12                                            deadline
13                                    state_changed_at
14                                          created_at
15                                         launched_at
16                  is_in_post_campaign_pledging_phase
17        

In [133]:
with pd.option_context('display.max_rows', None):
    print(pd.DataFrame(df_b.columns))

                              0
0                    Unnamed: 0
1                            id
2                         photo
3                          name
4                         blurb
5                          goal
6                       pledged
7                         state
8                          slug
9         disable_communication
10                      country
11                     currency
12              currency_symbol
13       currency_trailing_code
14                     deadline
15             state_changed_at
16                   created_at
17                  launched_at
18                   staff_pick
19                backers_count
20              static_usd_rate
21                  usd_pledged
22                      creator
23                     location
24                     category
25                      profile
26                    spotlight
27                         urls
28                   source_url
29                     name_len
30      

At a glance we can see that the `df_b` dataset has only half as many columns, and many of those columns are meta. (For example, `launched_at_day`, `launched_at_yr`, etc.)

Since the `df_b` dataset is already limited and it comes from a Kaggle project where the author has done one predictive project, let's go ahead and drop this dataset.

We'll just focus with df_a.

In [139]:
df = df_a_parse

# Cleaning Data

Let's go ahead and check each column and check its data cleanliness. If need be, we can drop columns as we go.
Because there are so many columns, we'll break the calls into smaller ranges of columns. 

In [144]:
df.iloc[:, 0:10].head(1)

,id,name,blurb,goal,pledged,state,slug,country,country_displayable_name,currency
0,1703332404,Massage By Donation... Seriously!,Physical wellness is not a luxury; it is a nec...,53170.0,52.0,failed,massage-by-donation-seriously,US,the United States,USD


In [145]:
df.iloc[:, 0:10].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112499 entries, 0 to 112498
Data columns (total 10 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   id                        112499 non-null  int64  
 1   name                      112499 non-null  object 
 2   blurb                     112499 non-null  object 
 3   goal                      112499 non-null  float64
 4   pledged                   112499 non-null  float64
 5   state                     112499 non-null  object 
 6   slug                      112499 non-null  object 
 7   country                   112499 non-null  object 
 8   country_displayable_name  112499 non-null  object 
 9   currency                  112499 non-null  object 
dtypes: float64(2), int64(1), object(7)
memory usage: 8.6+ MB


In [146]:
df.iloc[:, 0:10].describe()

,id,goal,pledged
count,1.124990e+05,1.124990e+05,1.124990e+05
mean,1.074855e+09,5.879430e+04,4.169411e+04
std,6.181879e+08,4.315790e+06,1.014663e+06
min,1.358300e+04,0.000000e+00,0.000000e+00
25%,5.399921e+08,1.000000e+03,8.100000e+01
50%,1.076940e+09,4.000000e+03,1.657000e+03
75%,1.607259e+09,1.000000e+04,7.678000e+03
max,2.147467e+09,1.000000e+09,1.755535e+08


The first `10` columns are okay.

In [147]:
df.iloc[:, 10:20].head(2)

,currency_symbol,currency_trailing_code,deadline,state_changed_at,created_at,launched_at,is_in_post_campaign_pledging_phase,staff_pick,is_starrable,disable_communication
0,$,True,1743442361,1743442365,1736541915,1738261961,False,False,False,False
1,€,False,1742639379,1742639380,1736248932,1737458979,False,False,False,False


We note that there are multiple currencies. We'll need to find a conversion method.

#### Update To Do

- Convert all currencies to USD, based on time period.

A little Google research tells us that the `is_in_post_campaign_pledging_phase` column refers to the campaign being completed, but still accepting late pledges. We don't need this data.

We also don't need to save data about `is_starrable`. This simply tells us what the logged-in user can do, star or not star. We'll drop these types of columns.

In [149]:
df = df.drop(columns=['is_in_post_campaign_pledging_phase', 'is_starrable'])

In [150]:
df.iloc[:, 10:18].head(2)

,currency_symbol,currency_trailing_code,deadline,state_changed_at,created_at,launched_at,staff_pick,disable_communication
0,$,True,1743442361,1743442365,1736541915,1738261961,False,False
1,€,False,1742639379,1742639380,1736248932,1737458979,False,False


In [151]:
df.iloc[:, 10:18].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112499 entries, 0 to 112498
Data columns (total 8 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   currency_symbol         112499 non-null  object
 1   currency_trailing_code  112499 non-null  bool  
 2   deadline                112499 non-null  int64 
 3   state_changed_at        112499 non-null  int64 
 4   created_at              112499 non-null  int64 
 5   launched_at             112499 non-null  int64 
 6   staff_pick              112499 non-null  bool  
 7   disable_communication   112499 non-null  bool  
dtypes: bool(3), int64(4), object(1)
memory usage: 4.6+ MB


In [152]:
df.iloc[:, 10:18].describe()

,deadline,state_changed_at,created_at,launched_at
count,1.124990e+05,1.124990e+05,1.124990e+05,1.124990e+05
mean,1.501233e+09,1.623366e+09,1.615847e+09,1.487630e+09
std,4.336887e+08,1.371571e+08,1.371362e+08,4.484821e+08
min,0.000000e+00,1.242468e+09,1.240456e+09,0.000000e+00
25%,1.457064e+09,1.494396e+09,1.487838e+09,1.450650e+09
50%,1.612105e+09,1.659025e+09,1.650546e+09,1.604440e+09
75%,1.742412e+09,1.751440e+09,1.743248e+09,1.738603e+09
max,1.790050e+09,1.784885e+09,1.784867e+09,1.784885e+09


A question to explore: If a project has its communications disabled (which is usually due to fraud-type activities), does that relate to the pledged amount?

In [153]:
df.iloc[:, 18:30].head(2)

,backers_count,static_usd_rate,usd_pledged,converted_pledged_amount,fx_rate,usd_exchange_rate,current_currency,usd_type,video,spotlight,percent_funded,is_liked
0,3,1.000000,52.0,52.0,1.000000,1.000000,USD,domestic,NaN,False,0.0978,None
1,0,1.030272,0.0,0.0,1.138244,1.087666,USD,domestic,NaN,False,0.0000,None


Excellent. We have a `static_usd_rate` column, and a number of other useful columns, which should give us the ability to see more clearly the values of the targets, pledges, and concluding amounts.

A question to explore: How do Kickstarter projects compare across different currencies?

In [154]:
df.iloc[:, 18:30].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112499 entries, 0 to 112498
Data columns (total 12 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   backers_count             112499 non-null  int64  
 1   static_usd_rate           112499 non-null  float64
 2   usd_pledged               103880 non-null  object 
 3   converted_pledged_amount  103880 non-null  float64
 4   fx_rate                   112499 non-null  float64
 5   usd_exchange_rate         103880 non-null  float64
 6   current_currency          112499 non-null  object 
 7   usd_type                  112499 non-null  object 
 8   video                     0 non-null       float64
 9   spotlight                 112499 non-null  bool   
 10  percent_funded            112499 non-null  float64
 11  is_liked                  0 non-null       object 
dtypes: bool(1), float64(6), int64(1), object(4)
memory usage: 9.5+ MB


The `video` and `is_liked` columns have no use, again.

In [155]:
df = df.drop(columns=['video', 'is_liked'])

In [156]:
df.iloc[:, 18:28].describe()

,backers_count,static_usd_rate,converted_pledged_amount,fx_rate,usd_exchange_rate,percent_funded
count,112499.000000,112499.000000,1.038800e+05,112499.000000,103880.000000,1.124990e+05
mean,117.436031,0.909060,1.424380e+04,0.980616,0.984243,2.737596e+03
std,744.093349,0.363179,2.057660e+05,0.257713,0.261566,6.441253e+05
min,0.000000,0.000000,0.000000e+00,0.006105,0.006105,0.000000e+00
25%,3.000000,1.000000,2.200000e+02,1.000000,1.000000,1.345500e+00
50%,25.000000,1.000000,2.050000e+03,1.000000,1.000000,1.027512e+02
75%,86.000000,1.000000,7.783250e+03,1.000000,1.000000,1.404117e+02
max,105857.000000,1.716408,4.676226e+07,1.331437,1.716408,2.149404e+08


On average, `117` backers support each campaign.

The mean value of `2738%` funding is unlikely to be relevant, because some campaigns are so successful they skew the dataset. The median of `103%` funded is more likely to be accurate.

In [158]:
df.iloc[:, 28:41].head(2)

,is_disliked,is_launched,prelaunch_activated,source_url,photo.key,photo.full,photo.ed,photo.med,photo.little,photo.small,photo.thumb,photo.1024x576,photo.1536x864
0,None,True,False,https://www.kickstarter.com/discover/categorie...,assets/047/794/907/72d3766ab073b662350a131c856...,https://i.kickstarter.com/assets/047/794/907/7...,https://i.kickstarter.com/assets/047/794/907/7...,https://i.kickstarter.com/assets/047/794/907/7...,https://i.kickstarter.com/assets/047/794/907/7...,https://i.kickstarter.com/assets/047/794/907/7...,https://i.kickstarter.com/assets/047/794/907/7...,https://i.kickstarter.com/assets/047/794/907/7...,https://i.kickstarter.com/assets/047/794/907/7...
1,None,True,False,https://www.kickstarter.com/discover/categorie...,assets/047/756/169/65ca76bc34cea486ebe7f36731c...,https://i.kickstarter.com/assets/047/756/169/6...,https://i.kickstarter.com/assets/047/756/169/6...,https://i.kickstarter.com/assets/047/756/169/6...,https://i.kickstarter.com/assets/047/756/169/6...,https://i.kickstarter.com/assets/047/756/169/6...,https://i.kickstarter.com/assets/047/756/169/6...,https://i.kickstarter.com/assets/047/756/169/6...,https://i.kickstarter.com/assets/047/756/169/6...


The `is_disliked` and all of the `photo...` columns are not needed. 

In [163]:
df = df.drop(columns=['is_disliked', 'photo.key', 'photo.full', 'photo.ed', 'photo.med', 'photo.little', 'photo.small', 'photo.thumb', 'photo.1024x576', 'photo.1536x864'])

In [164]:
df.iloc[:, 28:41].head(2)

,is_launched,prelaunch_activated,source_url,creator.id,creator.name,creator.slug,creator.is_registered,creator.is_email_verified,creator.chosen_currency,creator.is_superbacker,creator.has_admin_message_badge,creator.partner_badge,creator.ppo_has_action
0,True,False,https://www.kickstarter.com/discover/categorie...,85356126,Pressure Massage,pressuremassage,None,None,None,None,False,None,False
1,True,False,https://www.kickstarter.com/discover/categorie...,2075243650,Tobias Kaiser,eifel-abenteuer,None,None,None,None,False,None,False


Is the `prelaunch_activated` only for projects at that time? Or is it historical?

I'm guessing the former, which makes this column not useful.

In [165]:
df = df.drop(columns=['prelaunch_activated'])

The `source_url` gives us information about categories on Kickstarter. We can later parse this.

#### Update To Do

- Parse `source_url` for category information